# Is using `dask` on JASMIN impossible? Maybe!


In [1]:
import datetime as dt
import os
import sys
import warnings
from pathlib import Path

import cartopy.crs as ccrs
import cmocean as cmo
import dask
import dask_gateway
import easygems.healpix as egh
import healpy as hp
import intake
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from dask.distributed import progress

warnings.filterwarnings(
    "ignore",
    message=".*The return type of `Dataset.dims` will be changed.*",
    category=FutureWarning,
)

# Make local project modules importable. Adjust if this notebook is not one level
# below your project root.
project_root = Path.cwd().parent.resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils import hp_mods, hp_to_latlon
from onset_utils import onset_period_1d

print("imports done")
print(f"Project root: {project_root}")

In [40]:
# Create a connection to dask-gateway.
gw = dask_gateway.Gateway("https://dask-gateway.jasmin.ac.uk", auth="jupyterhub")

# Inspect and change the options if required before creating your cluster.
options = gw.cluster_options()
options.worker_cores = 4
options.worker_memory =16
options.worker_setup = """
source /home/users/franmorr/miniforge3/bin/activate
conda activate /home/users/franmorr/miniforge3/envs/hackathon
"""
options.account = "firstrains"

# Create a Dask cluster, or, if one already exists, connect to it.
# This stage creates the scheduler job in Slurm, so it may take some
# time while your job queues.
clusters = gw.list_clusters()
if not clusters:
    cluster = gw.new_cluster(options, shutdown_on_close=False)
else:
    cluster = gw.connect(clusters[0].name)

# Create at least one worker, and allow your cluster to scale to three.
cluster.adapt(minimum=1, maximum=3)

# Get a Dask client.
client = cluster.get_client()
client.upload_file("/home/users/franmorr/hk26/hackathon-monsoons/utils.py")
client.upload_file("/home/users/franmorr/hk26/hackathon-monsoons/onset_identification/OnsetPeriod_toolbox.py")
client.upload_file("/home/users/franmorr/hk26/hackathon-monsoons/onset_identification/onset_utils.py")

20241001T0000Z_engl_t+3_000                     hk25/
Miniforge3-Linux-x86_64.sh                      hk26/
README.anaconda_defaults_usage_franmorr.txt     hpc_exercises/
README.anaconda_defaults_usage_franmorr_v2.txt  kscale@
aew_tracking/                                   miniforge3/
badc@                                           misc_notebooks/
cccb/                                           onset-energetics/
chirps_data@                                    onset-metrics/
circ_budget/                                    sl.202210.daymean.nc
data/                                           sp.202210.daymean.nc
di396-r1a.pb20221003.pp                         swift@
drylines/                                       teaching/
firstrains@                                     uniq_cloudbands/
firstrains_code/                                venvs/
folder_to_make_gifs/                            warmstart/


In [41]:
client

In [ ]:
client.get_scheduler_logs()

In [46]:
max_periods=2
zoom=7
zooms=[7]
plot=False
outdir = (
    "/home/users/franmorr/hk26/hackathon-monsoons/onset_identification/onset_dates/"
)

In [20]:
url = 'https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml'
cat = intake.open_catalog(url)['online']
sim = "icon_d3hp003"
sim_cat = cat[sim]

In [21]:
ds = sim_cat(zoom=zoom).to_dask().pipe(egh.attach_coords)

In [22]:
            ds_latlon = hp_to_latlon(ds.pr, zoom)
            pp_latlon = ds_latlon.resample(time="1D").mean().chunk(dict(time=-1))
            pp_latlon *= 3600
            pp_latlon["units"] = "mm h-1"
            first_days, last_days = xr.apply_ufunc(
                onset_period_1d,
                pp_latlon,
                pp_latlon["time"],
                input_core_dims=[["time"], ["time"]],
                output_core_dims=[["period"], ["period"]],
                kwargs={
                    "max_periods": max_periods,
                    "max_dry_frac_rainfall": 0.1,
                    "refine": True,
                },
                vectorize=True,
                dask="parallelized",
                output_dtypes=[float, float],
                dask_gufunc_kwargs={
                    "output_sizes": {"period": max_periods},
                },
            )

            first_days = first_days.assign_coords(period=np.arange(max_periods))
            last_days = last_days.assign_coords(period=np.arange(max_periods))
            first_days = first_days.rename("first_day_of_period")
            last_days = last_days.rename("last_day_of_period")
            dwtps = xr.merge([first_days, last_days])
            dwtps.attrs.pop("hiopy::enable", None)
            for var in dwtps.variables:
                dwtps[var].attrs.pop("hiopy::enable", None)

In [47]:
for sim in [
    "ifs_tco3999-ng5_rcbmf_cf",
    "icon_d3hp003",
    "casesm2_10km_nocumulus",
    "nicam_gl11",
]:
    sim_cat = cat[sim]
    if plot:
        fig, axes = plt.subplots(
            len(zooms),
            2,
            # height_ratios=[1] * len(zooms) + [0.1],
            # figsize=(6, 10),
            subplot_kw={"projection": projection},
            layout="constrained",
        )
    label_ix = 0
    for zoom_ix, zoom in enumerate(zooms):
        print(sim, zoom)
        outfile = f"{outdir}/{sim}_zoom_{zoom}_dwtps.nc"
        if os.path.exists(outfile):
            print(f"{outfile} exists, skipping...")
            if plot:
                dwtps = xr.open_dataset(outfile)
            pass
        else:
            if "hk26" in sim:
                ds = sim_cat(zoom=zoom, time="PT1H").to_dask().pipe(hp_mods)
            else: 
                if "icon" in sim:
                    ds = sim_cat(zoom=zoom, time_method="inst", time="PT1H").to_dask().pipe(egh.attach_coords)
                else:
                    ds = sim_cat(zoom=zoom, time="PT1H").to_dask().pipe(egh.attach_coords)
            ds_latlon = hp_to_latlon(ds, zoom)
            pp_latlon = ds_latlon.pr.resample(time="1D").mean().chunk(dict(time=-1))
            pp_latlon *= 3600
            pp_latlon["units"] = "mm h-1"
            first_days, last_days = xr.apply_ufunc(
                onset_period_1d,
                pp_latlon,
                pp_latlon["time"],
                input_core_dims=[["time"], ["time"]],
                output_core_dims=[["period"], ["period"]],
                kwargs={
                    "max_periods": max_periods,
                    "max_dry_frac_rainfall": 0.1,
                    "refine": True,
                    # "precip_threshold": 0.05,
                    # "intensity_threshold": "60%",
                },
                vectorize=True,
                dask="parallelized",
                output_dtypes=[float, float],
                dask_gufunc_kwargs={
                    "output_sizes": {"period": max_periods},
                },
            )

            first_days = first_days.assign_coords(period=np.arange(max_periods))
            last_days = last_days.assign_coords(period=np.arange(max_periods))
            first_days = first_days.rename("first_day_of_period")
            last_days = last_days.rename("last_day_of_period")
            dwtps = xr.merge([first_days, last_days])
            dwtps.attrs.pop("hiopy::enable", None)
            for var in dwtps.variables:
                dwtps[var].attrs.pop("hiopy::enable", None)
            dwtps.to_netcdf(outfile)

        if plot:
            for ix in range(max_periods):
                label = labels[label_ix]
                ax = axes[zoom_ix, ix]
                ax.set_global()
                ax.coastlines()
                m = (dwtps.first_day_of_period.sel(period=ix) % 365).plot(
                    ax=ax, vmin=0, vmax=365, cmap=cmo.cm.phase, add_colorbar=False
                )

                ax.set_title(f"{label} zoom={zoom}")

                label_ix += 1
    if plot:
        fig.suptitle(sim)
        fig.colorbar(
            m,
            ax=axes.ravel().tolist(),
            orientation="horizontal",
            fraction=0.05,
            pad=0.07,
            shrink=0.7,
            label="day of year",
        )

        plt.savefig(f"images/{sim}_zooms_{''.join([str(zoom) for zoom in zooms])}.png")

In [ ]:
## I hereby conclude there is NO WAY to ever get this to work. Thanks and goodnight. 

In [30]:
client.get_versions(check=True)